# E-HLAstic Discovery of HLA-E Peptides from 3D12 Processed Jurkat Cell Line

### Define Global Parameters

In [1]:
import os

import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

project_dir = 'D:/Projects/ehlastic_release'

data_dir = os.path.join(project_dir, 'data')

fig_dir = os.path.join(project_dir, 'figure', 'Fig_Jurkat')


### Load Max's Validated HLA-E Binding Peptide Data

In [ ]:
Max_3D12_DF =  pd.read_excel(os.path.join(data_dir, "Max_Gerry", "Max_Tested_Peptides_CellReports.xlsx"), sheet_name="Jurkat_3D12")

# Set Threshold for Binding Prediction
Tm_Thresh = .5

Max_3D12_DF['Binder'] = (Max_3D12_DF['Tm gain'] > Tm_Thresh).astype(float)

### Compare the Three Predictor Results

In [3]:
# Combined 3-row figure: histograms (top), zoomed histograms (middle), scatter plots (bottom)
colors = {
    'EHLAstic': {1: '#b30000', 0: '#ffb3b3'},    # red
    'NetMHCpan': {1: '#00008b', 0: '#add8e6'},  # blue
    'MHCFlurry': {1: '#006400', 0: '#98fb98'}   # green
}

predictors = [
    ('eHLAstic_Normalized_Rank', 'EHLAstic', 'Normalized EHLAstic Score'),
    ('NetMHCpan_Rank_EL_HLA-E*01:03', 'NetMHCpan', 'NetMHCpan EL Rank'),
    ('MHCflurry_Presentation_Percentile', 'MHCFlurry', 'MHCFlurry Presentation Percentile')
]

short_titles = ['EHLAstic Score', 'NetMHCpan EL Rank', 'MHCFlurry Percentile']

# prepare data
ehldata = (1 - Max_3D12_DF['eHLAstic_Binding_Probability'].dropna()) * 100

mfhdata = Max_3D12_DF['MHCflurry_Presentation_Percentile'].dropna()

netdata = Max_3D12_DF['NetMHCpan_Rank_EL_HLA-E*01:03'].dropna()

datasets = [
    (ehldata, '#b30000', (0.0, 100.0), 2.0, 'EHLAstic'),
    (netdata, '#00008b', (0.0, 100.0), 2.0, 'NetMHCpan'),
    (mfhdata, '#006400', (0.0, 100.0), 2.0, 'MHCFlurry'),
]

# Panel labels per row
row_labels = [['A', 'B', 'C'], ['D', 'E', 'F'], ['G', 'H', 'I']]

os.makedirs(fig_dir, exist_ok=True)
fname = os.path.join(fig_dir, 'jurkat_3d12_predictor_histograms_combined')

fig, axes = plt.subplots(3, 3, figsize=(11, 10), sharey=False, constrained_layout=True)
fig.suptitle("Jurkat, 3D12", fontsize=14)
fig.set_constrained_layout_pads(w_pad=0.10, h_pad=0.08, wspace=0.12, hspace=0.16)

# Top row: histograms (full range 0-100)
for i, (data, color, xlim, cutoff, key) in enumerate(datasets):
    ax = axes[0, i]
    lbl = row_labels[0][i]
    bins = min(60, max(10, int(len(data) / 2) + 10))
    ax.hist(data, bins=bins, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    if len(data) > 3:
        try:
            sns.kdeplot(data, ax=ax, color='k', lw=1.0, bw_method='scott', alpha=0.6)
        except Exception:
            pass
    ax.set_title(f"{lbl}   {short_titles[i]} (n={len(data)})", loc='left', fontsize=10)
    ax.set_xlim(*xlim)
    ax.set_xlabel(short_titles[i])
    ax.set_ylabel('Count')
    ax.tick_params(axis='both', which='major', length=4, direction='out')
    sns.despine(ax=ax, trim=True)
    if cutoff is not None:
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        offset = (xlim[1] - xlim[0]) * 0.07
        ax.text(cutoff + offset, ylim[1] * 1.03, f'Cutoff = {cutoff}', rotation=90,
                ha='right', va='top', color='0.3', fontsize=8, backgroundcolor='white')
        ax.axvline(cutoff, color='0.3', linestyle='--', linewidth=1, zorder=10)

# Middle row: histograms (zoomed to 0-2.5)
for i, (data, color, _, cutoff, key) in enumerate(datasets):
    ax = axes[1, i]
    lbl = row_labels[1][i]
    zoomed_data = data[data <= 2.5]
    bins = min(60, max(10, int(len(zoomed_data) / 2) + 10))
    ax.hist(zoomed_data, bins=bins, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    if len(zoomed_data) > 3:
        try:
            sns.kdeplot(zoomed_data, ax=ax, color='k', lw=1.0, bw_method='scott', alpha=0.6)
        except Exception:
            pass
    ax.set_title(f"{lbl}   {short_titles[i]}, Zoomed (n={len(zoomed_data)})", loc='left', fontsize=10)
    ax.set_xlim(0.0, 2.5)
    ax.set_xlabel(short_titles[i])
    ax.set_ylabel('Count')
    ax.tick_params(axis='both', which='major', length=4, direction='out')
    sns.despine(ax=ax, trim=True)
    if cutoff is not None:
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        offset = (xlim[1] - xlim[0]) * 0.07
        ax.text(cutoff + offset, ylim[1] * 1.03, f'Cutoff = {cutoff}', rotation=90,
                ha='right', va='top', color='0.3', fontsize=8, backgroundcolor='white')
        ax.axvline(cutoff, color='0.3', linestyle='--', linewidth=1, zorder=10)

# Bottom row: swarm plots (predictor scores by Binder vs Non-binder)
for i, (col, key, ylabel) in enumerate(predictors):
    ax = axes[2, i]
    lbl = row_labels[2][i]
    df = Max_3D12_DF[[col, 'Binder']].dropna(subset=[col, 'Binder']).copy()
    if df.empty:
        ax.set_visible(False)
        continue
    df['Binding Status'] = df['Binder'].map({0: 'Non-binder', 1: 'Binder'})
    palette = {'Non-binder': colors[key][0], 'Binder': colors[key][1]}
    try:
        sns.swarmplot(
            data=df,
            x='Binding Status',
            y=col,
            hue='Binding Status',
            order=['Non-binder', 'Binder'],
            hue_order=['Non-binder', 'Binder'],
            palette=palette,
            dodge=False,
            legend=False,
            warn_thresh=0,
            size=4,
            linewidth=0,
            ax=ax
        )
    except Exception:
        sns.stripplot(
            data=df,
            x='Binding Status',
            y=col,
            order=['Non-binder', 'Binder'],
            palette=palette,
            dodge=False,
            size=4,
            jitter=0.25,
            ax=ax
        )
    ax.set_title(f"{lbl}   {short_titles[i]}, Binder vs Non-binder", loc='left', fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 2)
    ax.grid(alpha=0.25)
    sns.despine(ax=ax)
    handles = [
        mpl.patches.Patch(facecolor=colors[key][1], edgecolor='k', label='Binder'),
        mpl.patches.Patch(facecolor=colors[key][0], edgecolor='k', label='Non-binder')
    ]
    loc = 'upper right' if i < 2 else 'lower left'
    ax.legend(handles=handles, loc=loc, fontsize=8, frameon=False)

fig.savefig(f"{fname}.pdf", bbox_inches='tight')
fig.savefig(f"{fname}.png", dpi=600, bbox_inches='tight')
plt.close(fig)